<a href="https://colab.research.google.com/github/va2305/Bias_vs_Variance_Visualization/blob/main/toddler_chat_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Install & Import Required libraries

!pip install SpeechRecognition==3.10.0 gtts IPython pygame

import pandas as pd
import numpy as np
import random
import io
import time

import speech_recognition as sr
from gtts import gTTS
from IPython.display import Audio, display
from google.colab import files
import matplotlib.pyplot as plt
import difflib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.8/32.8 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1


In [ ]:
#CVC word
words_data = {
    "word": ['cat','dog','bat','hat','rat','mat','sat','fat','pat','man',
             'pan','can','fan','ran','tan','van','jam','ram','ham','yam',
             'pen','hen','ten','men','den','bed','red','fed','led','wed',
             'pig','dig','big','fig','wig','rug','bug','hug','jug','mug',
             'sun','fun','gun','run','bun','nun','pun','cup','pup','sup']
}
df = pd.DataFrame(words_data)
print("📚 50 CVC Words Loaded:")
print(df.head())


📚 50 CVC Words Loaded:
  word
0  cat
1  dog
2  bat
3  hat
4  rat


In [ ]:
#TTS - Bot Speaks
def speak_word(word, lang="en"):
    tts = gTTS(text=word.upper(), lang=lang, slow=False)
    audio_file = io.BytesIO()
    tts.write_to_fp(audio_file)
    audio_file.seek(0)
    return audio_file



In [ ]:
#Kids upload recording
r = sr.Recognizer()

def listen_from_file():
    print("🔊 Please upload your recording (say the word).")
    uploaded = files.upload()   # Kid/you record on phone & upload
    fname = list(uploaded.keys())[0]
    with sr.AudioFile(fname) as source:
        audio = r.record(source)
    try:
        text = r.recognize_google(audio).lower().strip()
        print(f"🎤 Heard: {text}")
        return text
    except sr.UnknownValueError:
        print("❌ Could not understand audio.")
        return ""
    except sr.RequestError as e:
        print("❌ API error:", e)
        return ""


In [ ]:
# Cell 4: 🎤 LIVE MICROPHONE (No Uploads!)
from IPython.display import Javascript
from google.colab import output
import base64

# JavaScript for mic recording (3 seconds)
RECORD_JS = """
const sleep  = time => new Promise(resolve => setTimeout(resolve,time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = () => resolve(reader.result)
  reader.readAsDataURL(blob)
})

async function record(sec=3){
  const mimeType = 'audio/webm'
  let stream = await navigator.mediaDevices.getUserMedia({audio:true})
  let recorder = new MediaRecorder(stream, {mimeType})
  let data = []

  recorder.ondataavailable = event => data.push(event.data)
  recorder.start()
  await sleep(sec*1000)
  recorder.stop()
  await sleep(1000)  // Wait for stop
  stream.getTracks().forEach(track => track.stop())

  let blob = new Blob(data, {type:mimeType})
  return await b2text(blob)
}
"""

display(Javascript(RECORD_JS))

r = sr.Recognizer()

def listen_live():
    print("🎤 CLICK 'RUN' → Allow mic → SPEAK word NOW (3 sec)")
    audio_b64 = output.eval_js('record(3)')

    # Decode base64 to bytes
    header, data = audio_b64.split(',')
    audio_bytes = base64.b64decode(data)

    # Convert to WAV for SpeechRecognition
    with sr.AudioFile(io.BytesIO(audio_bytes)) as source:
        audio = r.record(source)

    try:
        text = r.recognize_google(audio).lower().strip()
        print(f"🎤 Kid said: '{text}'")
        return text
    except:
        print("❌ Could not hear clearly.")
        return ""


<IPython.core.display.Javascript object>

In [ ]:
# 🎨 COMPLETE KID BOT - LIVE MIC + GUI (No JS Errors!)
!pip install ipywidgets -q
from ipywidgets import Button, VBox, HBox, HTML, Output
from IPython.display import display, clear_output, Javascript
import base64
import io

class ToddlerBot:
    def __init__(self):
        self.score = 0
        self.rounds = 0
        self.output = Output()
        self.df = df  # Your words_df

    def speak_word(self, word):
        from gtts import gTTS
        tts = gTTS(text=word, lang='en', slow=False)
        fp = io.BytesIO()
        tts.write_to_fp(fp)
        fp.seek(0)
        display(Audio(fp.read(), autoplay=True))

    def stt_live(self):
        """Fixed live mic - no JS scope issues"""
        js = Javascript('''
        async function recordMic() {
            const stream = await navigator.mediaDevices.getUserMedia({audio:true});
            const recorder = new MediaRecorder(stream);
            const chunks = [];
            return new Promise(resolve => {
                recorder.ondataavailable = e => chunks.push(e.data);
                recorder.onstop = async () => {
                    const blob = new Blob(chunks, {type: 'audio/webm'});
                    const reader = new FileReader();
                    reader.onload = () => {
                        stream.getTracks().forEach(t => t.stop());
                        resolve(reader.result.split(',')[1]);
                    }
                    reader.readAsDataURL(blob);
                }
                recorder.start();
                setTimeout(() => recorder.stop(), 3000);
            });
        }
        recordMic();
        ''')
        display(js)

        audio_b64 = output.eval_js('')
        if not audio_b64:
            return ""

        try:
            audio_bytes = base64.b64decode(audio_b64)
            r = sr.Recognizer()
            with sr.AudioFile(io.BytesIO(audio_bytes)) as source:
                audio = r.record(source)
            return r.recognize_google(audio).lower().strip()
        except:
            return ""

    def on_speak(self, b):
        word = random.choice(self.df['word']).upper()
        self.current_word = word

        with self.output:
            clear_output()
            print(f"🤖 Say: *** {word} ***")
            self.speak_word(word)

            spoken = self.stt_live()
            sim = difflib.SequenceMatcher(None, spoken, word.lower()).ratio()
            is_ok = sim >= 0.6

            self.score += 1 if is_ok else 0
            face = "🎉🥳" if is_ok else "💕😊"

            print(f"🎤 You: '{spoken}' ({sim:.0%})")
            print(f"Score: {self.score}")
            display(HTML(f"<h1 style='font-size:100px'>{face}</h1>"))

    def start(self):
        display(HTML("""
        <div style='font-size:60px; text-align:center; background:#FF9A9B; padding:40px; border-radius:25px; color:white'>
            <h1>🤖 Toddler Word Bot! 🎈</h1>
            <p style='font-size:50px'>Click to learn words! 👶</p>
        </div>
        """))

        btn = Button(description="🎤 SPEAK & LEARN!",
                    button_style='success',
                    icon='microphone',
                    layout={'width':'400px', 'height':'100px', 'font_size':'30px'})
        btn.on_click(self.on_speak)

        display(VBox([self.output, btn]))
        print("👶 Click button → Hear word → SPEAK into mic → See score!")

# 🚀 LAUNCH BOT
bot = ToddlerBot()
bot.start()


HTML(value="\n        <div style='font-size:60px; text-align:center; background:#FF9A9B; padding:40px; border-…

👶 Click button → Hear word → SPEAK into mic → See score!
